In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *

# Detect free RAM now, cap this kernel so it can never exhaust the machine, and
# make every parquet read/write below stream in bounded batches instead of
# loading the whole (~14 GB float64) frame at once.
install_memory_guard()

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

# PREDISPATCH forecast demand curve. predispatch_totaldemand_{region}_h{k} is the
# forecast total demand for the k-th 30-min period after each timestamp, from the
# run available at that timestamp. h1 ~= +30min ... h78 ~= +39h. Ex-ante
# forecasts, leakage-free by construction, consumed unshifted.
PD_DEMAND_PREFIX = "predispatch_totaldemand"
PD_HORIZONS      = 78

SRC_PATH = "../1_Dataset/Processed_data/6_2_predispatch_regionsum.parquet"
OUT_PATH = "../2_Features_build/Feature_data/6_2_predispatch_region_sum.parquet"


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.2G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


In [2]:
# The source is ~1951 float64 columns; loading it whole is what crashed the
# kernel. The final cell streams it in bounded row-batches straight to disk. Here
# we only pull a tiny sample so the feature functions below can be previewed
# without ever holding the full frame in memory.
sample = peek_parquet(SRC_PATH, 10)
sample.iloc[:, :6]


,predispatch_totaldemand_nsw_h1,predispatch_totaldemand_nsw_h2,predispatch_totaldemand_nsw_h3,predispatch_totaldemand_nsw_h4,predispatch_totaldemand_nsw_h5,predispatch_totaldemand_nsw_h6
Date,,,,,,
2018-01-01 00:00:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:05:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:10:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:15:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:20:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:25:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863
2018-01-01 00:30:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098
2018-01-01 00:35:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098
2018-01-01 00:40:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098


In [3]:
def _add_predispatch_demand_curve(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expose the target region's full predispatch demand forecast curve. Each
    horizon k aligns to a future 30-min delivery period, giving the model the
    expected load trajectory that drives the price. Leakage-free, used unshifted.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    for k in range(1, PD_HORIZONS + 1):
        c = f"{PD_DEMAND_PREFIX}_{R}_h{k}"
        if c in df.columns:
            new_cols[f"pd_demand_{R}_fh{k}"] = df[c].astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_demand_curve(sample)[:10]


,pd_demand_nsw_fh1,pd_demand_nsw_fh2,pd_demand_nsw_fh3,pd_demand_nsw_fh4,pd_demand_nsw_fh5,pd_demand_nsw_fh6,pd_demand_nsw_fh7,pd_demand_nsw_fh8,pd_demand_nsw_fh9,pd_demand_nsw_fh10,...,pd_demand_nsw_fh69,pd_demand_nsw_fh70,pd_demand_nsw_fh71,pd_demand_nsw_fh72,pd_demand_nsw_fh73,pd_demand_nsw_fh74,pd_demand_nsw_fh75,pd_demand_nsw_fh76,pd_demand_nsw_fh77,pd_demand_nsw_fh78
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:05:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:10:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:15:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:20:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:25:00,6900.839844,6685.419922,6401.779785,6185.609863,6040.169922,5937.109863,5876.879883,5838.770020,5851.120117,5821.089844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:30:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098,5859.939941,5873.830078,5844.160156,5906.839844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:35:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098,5859.939941,5873.830078,5844.160156,5906.839844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039
2018-01-01 00:40:00,6691.149902,6408.790039,6195.979980,6055.689941,5954.529785,5896.100098,5859.939941,5873.830078,5844.160156,5906.839844,...,6162.22998,6157.009766,6158.5,6163.220215,6163.180176,6159.609863,6159.580078,6160.799805,6160.759766,6160.790039


In [4]:
def _add_predispatch_demand_shape(df: pd.DataFrame) -> pd.DataFrame:
    """
    Shape statistics of the target region's forecast demand curve: expected
    level, peak and intraday range over the next 6h / 24h / 39h, plus the slope
    from the near to the far end. A large forecast peak or range flags an
    approaching demand-driven price event. Leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    windows = [("6h", range(1, 13)), ("24h", range(1, 49)), ("39h", range(1, PD_HORIZONS + 1))]
    for lab, ks in windows:
        cc = [f"{PD_DEMAND_PREFIX}_{R}_h{k}" for k in ks if f"{PD_DEMAND_PREFIX}_{R}_h{k}" in df.columns]
        sub = df[cc]
        new_cols[f"pd_demand_{R}_fmean_{lab}"]  = sub.mean(axis=1).astype(np.float32)
        new_cols[f"pd_demand_{R}_fmax_{lab}"]   = sub.max(axis=1).astype(np.float32)
        new_cols[f"pd_demand_{R}_frange_{lab}"] = (sub.max(axis=1) - sub.min(axis=1)).astype(np.float32)

    early = df[[f"{PD_DEMAND_PREFIX}_{R}_h{k}" for k in range(1, 7)]].mean(axis=1)
    late  = df[[f"{PD_DEMAND_PREFIX}_{R}_h{k}" for k in range(PD_HORIZONS - 5, PD_HORIZONS + 1)]].mean(axis=1)
    new_cols[f"pd_demand_{R}_fslope"] = (late - early).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_demand_shape(sample)[:10]


,pd_demand_nsw_fmean_6h,pd_demand_nsw_fmax_6h,pd_demand_nsw_frange_6h,pd_demand_nsw_fmean_24h,pd_demand_nsw_fmax_24h,pd_demand_nsw_frange_24h,pd_demand_nsw_fmean_39h,pd_demand_nsw_fmax_39h,pd_demand_nsw_frange_39h,pd_demand_nsw_fslope
Date,,,,,,,,,,
2018-01-01 00:00:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:05:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:10:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:15:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:20:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:25:00,6111.971191,6900.839844,1079.750000,7238.278809,8712.429688,2891.339844,6846.057617,8712.429688,2891.339844,-197.701172
2018-01-01 00:30:00,6057.552246,6691.149902,846.989746,7259.789062,8741.730469,2897.570312,6849.885254,8741.730469,2897.570312,-39.585938
2018-01-01 00:35:00,6057.552246,6691.149902,846.989746,7259.789062,8741.730469,2897.570312,6849.885254,8741.730469,2897.570312,-39.585938
2018-01-01 00:40:00,6057.552246,6691.149902,846.989746,7259.789062,8741.730469,2897.570312,6849.885254,8741.730469,2897.570312,-39.585938


In [5]:
def _add_predispatch_demand_neighbour(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compact forecast-demand context for the other regions (expected level and
    peak over the next 24h). Coincident regional demand peaks compete for the
    same interconnector capacity and lift the target region's price. Leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    for r in OTHER_REGIONS:
        cc = [f"{PD_DEMAND_PREFIX}_{r}_h{k}" for k in range(1, 49) if f"{PD_DEMAND_PREFIX}_{r}_h{k}" in df.columns]
        if cc:
            sub = df[cc]
            new_cols[f"pd_demand_{r}_fmean_24h"] = sub.mean(axis=1).astype(np.float32)
            new_cols[f"pd_demand_{r}_fmax_24h"]  = sub.max(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_predispatch_demand_neighbour(sample)[:10]


,pd_demand_qld_fmean_24h,pd_demand_qld_fmax_24h,pd_demand_vic_fmean_24h,pd_demand_vic_fmax_24h,pd_demand_sa_fmean_24h,pd_demand_sa_fmax_24h
Date,,,,,,
2018-01-01 00:00:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:05:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:10:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:15:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:20:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:25:00,6446.134766,7761.560059,3921.841553,4470.009766,1003.633118,1353.849976
2018-01-01 00:30:00,6453.557129,7751.779785,3905.251953,4463.109863,1004.192139,1353.939941
2018-01-01 00:35:00,6453.557129,7751.779785,3905.251953,4463.109863,1004.192139,1353.939941
2018-01-01 00:40:00,6453.557129,7751.779785,3905.251953,4463.109863,1004.192139,1353.939941


In [6]:
def add_predispatch_features(df: pd.DataFrame) -> pd.DataFrame:
    # All three groups are row-wise over the horizon columns, so they compute
    # identically on a row-batch as on the full frame.
    return pd.concat(
        [
            _add_predispatch_demand_curve(df),
            _add_predispatch_demand_shape(df),
            _add_predispatch_demand_neighbour(df),
        ],
        axis=1,
    )


# Retain core columns (keep_source=True): the predispatch demand forecast curve
# is an ex-ante forecast from the run available at t -> leakage-free, used
# unshifted. Streams source + new features to disk in bounded batches; peak RAM
# is one batch, so this cannot exhaust memory regardless of file size.
stream_transform_parquet(SRC_PATH, OUT_PATH, transform=add_predispatch_features, keep_source=True)

import pyarrow.parquet as pq
meta = pq.ParquetFile(OUT_PATH).metadata
print("Total features:", meta.num_columns)
(meta.num_rows, meta.num_columns)


[stream] 6_2_predispatch_regionsum.parquet: 893,665 rows x 1951 cols, largest row group 2.09G -> column-block strategy (bounded regardless of free RAM)


Merge rows: 100%|██████████| 104/104 [01:05<00:00,  1.59batch/s]


[stream] wrote 893,665 rows -> ../2_Features_build/Feature_data/6_2_predispatch_region_sum.parquet
Total features: 2045


(893665, 2045)

In [7]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 14 variable(s); kernel rss 0.52G, 9.4G RAM free now
